In [1]:
# ML Model 3 — Length of Stay Prediction

## Objective
Predict the number of days a patient stays in the hospital.

## Problem Type
Regression

## Target Variable
length_of_stay

## Evaluation Metrics
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score

SyntaxError: invalid character '²' (U+00B2) (1695311477.py, line 15)

Dataset shape: (50000, 11)

Columns:
['admission_id', 'patient_id', 'department_id', 'admission_date', 'discharge_date', 'admission_type', 'age', 'gender', 'length_of_stay', 'bed_type', 'discharge_status']

Target: length_of_stay

Length-of-Stay Statistics:
count    50000.000000
mean         3.818100
std          3.152093
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         44.000000
Name: length_of_stay, dtype: float64

Training data: (40000, 7)
Testing data: (10000, 7)


KeyboardInterrupt: 

In [6]:
# ML Model 3 — Length-of-Stay Prediction
# Faster version for our 50,000-row dataset.
# We use a smaller Random Forest so we can evaluate the model quickly.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================================
# 1. LOAD DATA
# ==========================================================

file_path = "../data/processed/admissions/admissions_clean.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)


# ==========================================================
# 2. CREATE DATE FEATURES
# ==========================================================

df["admission_date"] = pd.to_datetime(df["admission_date"])

df["admission_dayofweek"] = df["admission_date"].dt.dayofweek
df["admission_month"] = df["admission_date"].dt.month


# ==========================================================
# 3. TARGET
# ==========================================================

target = "length_of_stay"

y = df[target]

print("\nTarget:", target)
print("\nLength-of-Stay Statistics:")
print(y.describe())


# ==========================================================
# 4. FEATURES
# ==========================================================

features = [
    "department_id",
    "admission_type",
    "age",
    "gender",
    "bed_type",
    "admission_dayofweek",
    "admission_month"
]

X = df[features]


# ==========================================================
# 5. FEATURE TYPES
# ==========================================================

categorical_features = [
    "department_id",
    "admission_type",
    "gender",
    "bed_type"
]

numerical_features = [
    "age",
    "admission_dayofweek",
    "admission_month"
]


# ==========================================================
# 6. PREPROCESSING
# ==========================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# ==========================================================
# 7. TRAIN / TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# ==========================================================
# 8. FASTER RANDOM FOREST
# ==========================================================

model = RandomForestRegressor(
    n_estimators=30,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)


# ==========================================================
# 9. PIPELINE
# ==========================================================

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)


# ==========================================================
# 10. TRAIN
# ==========================================================

print("\nTraining Length-of-Stay model...")

model_pipeline.fit(X_train, y_train)

print("Training completed successfully!")


# ==========================================================
# 11. PREDICTIONS
# ==========================================================

print("\nGenerating predictions...")

y_pred = model_pipeline.predict(X_test)

print("Predictions generated successfully!")


# ==========================================================
# 12. EVALUATION
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)


print("\n========== LENGTH-OF-STAY MODEL PERFORMANCE ==========")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ==========================================================
# 13. SAMPLE PREDICTIONS ONLY
# ==========================================================

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\n========== SAMPLE PREDICTIONS ==========")
print(results.head(10))

Dataset shape: (50000, 11)

Target: length_of_stay

Length-of-Stay Statistics:
count    50000.000000
mean         3.818100
std          3.152093
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         44.000000
Name: length_of_stay, dtype: float64

Training data: (40000, 7)
Testing data: (10000, 7)

Training Length-of-Stay model...
Training completed successfully!

Generating predictions...
Predictions generated successfully!

========== LENGTH-OF-STAY MODEL PERFORMANCE ==========
MAE  : 2.0832
RMSE : 2.8240
R²   : 0.1496

========== SAMPLE PREDICTIONS ==========
   Actual  Predicted
0       2   4.056692
1       1   4.119983
2       2   2.380007
3       2   2.345155
4       4   3.457756
5      14   9.141291
6       1   3.359889
7       2   3.945304
8       3   4.685753
9       4   5.521832


In [7]:
# Display only the three important evaluation metrics.

print("========== LENGTH-OF-STAY MODEL PERFORMANCE ==========")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

========== LENGTH-OF-STAY MODEL PERFORMANCE ==========
MAE  : 2.0832
RMSE : 2.8240
R²   : 0.1496
